In [ ]:
from eval import tracs_rate_scenario, show_comparison_study, show_all_relations, compute_argoverse2_metrics
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
data_root = Path("../data")
backbone = "DeMo"
number_of_tests = 10

backbone_results_path = data_root / "av2_backbone_outputs" / backbone
assert backbone_results_path.exists(), (
    f"Backbone results path {backbone_results_path} does not exist."
)

In [ ]:
def get_worst(metrics_path: str, n_head=36):
    metrics_df = pd.read_parquet(metrics_path)
    print("before filtering:", len(metrics_df))
    metrics_df = metrics_df[
        (metrics_df["is_missed_prediction1"]) & (~metrics_df["is_missed_prediction6"])
    ]
    print("after filtering:", len(metrics_df))
    return metrics_df.sort_values("brier_minADE6", ascending=False).head(n_head)


worst = get_worst(backbone_results_path / "metrics.parquet")

In [ ]:
logic = """
distance(vehicle) <- source("/star_map/distance/vehicle", Density).
distance(focal_lane) <- source("/star_map/distance/focal_lane", Density).
speed <- source("/sensors/speed", Number).

# safe if vehicle(V) and distance(V) > 10.

unsafe if vehicle(V) and distance(V) < 10.
landscape if not unsafe.

# landscape if distance(focal_lane) < 1.

landscape -> target("/landscape").
"""

for i in range(number_of_tests):
    scenario_id = worst.iloc[i]["scenario_id"]
    scored_predictions, gt_score = tracs_rate_scenario(
        scenario_id=scenario_id,
        logic=logic,
        prediction_path=backbone_results_path / "submission_test.parquet",
        data_root=data_root,
        split="test",
    )
    show_comparison_study(
        scored_predictions=scored_predictions,
        gt_score=gt_score,
        prediction_path=backbone_results_path / "submission_test.parquet",
        data_root=data_root,
        split="test",
        scenario_id=scenario_id,
    )
    show_all_relations(
        scenario_id=scenario_id,
        logic=logic,
        prediction_path=backbone_results_path / "submission_test.parquet",
        data_root=data_root,
        split="test",
    )

    metrics = compute_argoverse2_metrics(scored_predictions, data_root, "test", scenario_id)
    melted = metrics.melt("method")
    melted = melted[melted["variable"] != "scenario_id"]
    melted = melted[melted["variable"] != "track_id"]
    melted = melted[melted["variable"] != "is_missed_prediction6"]
    melted = melted[melted["variable"] != "is_missed_prediction1"]

    plt.title(f"{scenario_id}")
    sns.barplot(melted, x="variable", y="value", hue="method")
    plt.xticks(rotation=30)
    plt.show()